# 🚗 Complete Vehicle Detection Training
## Brand + Model + Type + Attributes

**Extracts maximum info from single photo:**
- Brand (32 Mexican market brands)
- Model (specific models per brand)
- Car Type (sedan, SUV, hatchback, etc.)
- Year (from directory structure)

**⚠️ IMPORTANT:** Ve a **Settings > Internet > ON**

In [ ]:
!pip install -q tqdm scipy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
import time
import os
import scipy.io as sio
import random
import shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Find dataset
import zipfile

dataset_path = '/kaggle/input/datasets/ricardominor/vehicle-data'
print(f'Dataset path: {dataset_path}')
print(f'Contents: {os.listdir(dataset_path)}')

# Check if extracted
extracted_path = f'{dataset_path}/compcar/extracted/data'
if not os.path.exists(extracted_path):
    # Try to find and extract zip
    for item in os.listdir(dataset_path):
        if item.endswith('.zip'):
            print(f'\nExtracting {item}...')
            with zipfile.ZipFile(f'{dataset_path}/{item}', 'r') as z:
                z.extractall('/kaggle/working/')
            extracted_path = '/kaggle/working/compcar/extracted/data'
            break

if os.path.exists(extracted_path):
    print(f'\nCompCar ready at: {extracted_path}')
    print(f'Contents: {os.listdir(extracted_path)}')
else:
    print('ERROR: CompCar not found!')

In [ ]:
# Load ALL metadata
mat_path = f'{extracted_path}/misc/make_model_name.mat'
mat = sio.loadmat(mat_path)

# Brands
brands = {}
for idx, make in enumerate(mat['make_names']):
    if len(make) > 0 and len(make[0]) > 0:
        brand_name = make[0][0]
        if isinstance(brand_name, str):
            brands[idx] = brand_name

# Models
models_dict = {}
for idx, model in enumerate(mat['model_names']):
    if len(model) > 0 and len(model[0]) > 0:
        model_name = model[0][0]
        if isinstance(model_name, str):
            models_dict[idx] = model_name

# Car types
car_type_mat = sio.loadmat(f'{extracted_path}/misc/car_type.mat')
car_types = {}
if 'types' in car_type_mat:
    for i, t in enumerate(car_type_mat['types']):
        if len(t) > 0:
            car_types[i] = t[0]

# Attributes
attributes = {}
with open(f'{extracted_path}/misc/attributes.txt') as f:
    header = f.readline()  # Skip header
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 6:
            model_id = int(parts[0])
            attributes[model_id] = {
                'max_speed': int(parts[1]),
                'displacement': float(parts[2]),
                'doors': int(parts[3]),
                'seats': int(parts[4]),
                'type': int(parts[5])
            }

print(f'Brands: {len(brands)}')
print(f'Models: {len(models_dict)}')
print(f'Car types: {len(car_types)}')
print(f'Attributes: {len(attributes)}')

# Show car types
print(f'\nCar types available:')
for idx, name in car_types.items():
    count = sum(1 for a in attributes.values() if a['type'] == idx)
    print(f'  {idx}: {name} ({count} models)')

In [ ]:
# Mexican brands mapping
MEXICAN_BRANDS = {
    "Acura": "Acura", "Audi": "Audi", "BYD": "BYD",
    "Changan Business": "Changan", "Dodge": "Dodge", "FIAT": "Fiat",
    "Ford": "Ford", "GAC": "GAC", "GMC": "GMC", "Geely": "Geely",
    "Honda": "Honda", "Hyundai ": "Hyundai", "Infiniti": "Infiniti",
    "Jeep": "Jeep", "KIA": "Kia", "Lexus": "Lexus", "Lincoln": "Lincoln",
    "MG": "MG", "MAZDA": "Mazda", "Benz": "Mercedes-Benz",
    "MINI": "Mini", "Mitsubishi": "Mitsubishi", "Nissan": "Nissan",
    "Peugeot": "Peugeot", "Renault": "Renault", "Seat": "SEAT",
    "Subaru": "Subaru", "Suzuki": "Suzuki", "TESLA": "Tesla",
    "Toyota": "Toyota", "Volkswagen": "Volkswagen", "Volvo": "Volvo",
}

# Find Mexican brands
brand_mapping = {}
for brand_id, compcar_name in brands.items():
    for compcar_key, our_name in MEXICAN_BRANDS.items():
        if compcar_name.strip() == compcar_key.strip():
            brand_mapping[brand_id] = {
                "compcar_name": compcar_name,
                "our_name": our_name,
            }
            break

print(f'Found {len(brand_mapping)} Mexican market brands')

# Prepare comprehensive dataset
output_dir = '/kaggle/working/vehicle_complete'
os.makedirs(output_dir, exist_ok=True)

total_models = 0
total_images = 0
model_data = []  # Store metadata for each model

for brand_id, brand_info in brand_mapping.items():
    brand_name = brand_info['our_name']
    brand_dir = f'{extracted_path}/image/{brand_id}'
    
    if not os.path.exists(brand_dir):
        continue
    
    for model_dir_name in os.listdir(brand_dir):
        model_dir = f'{brand_dir}/{model_dir_name}'
        if not os.path.isdir(model_dir):
            continue
        
        model_id = int(model_dir_name)
        model_name = models_dict.get(model_id, f'Unknown_{model_id}')
        
        # Get attributes
        attrs = attributes.get(model_id, {})
        car_type_id = attrs.get('type', 0)
        car_type_name = car_types.get(car_type_id, 'Desconocido')
        
        # Count images
        model_images = []
        for year_dir in os.listdir(model_dir):
            year_path = f'{model_dir}/{year_dir}'
            if os.path.isdir(year_path):
                for img in os.listdir(year_path):
                    if img.endswith('.jpg'):
                        model_images.append(f'{year_path}/{img}')
        
        if len(model_images) < 10:
            continue
        
        # Create model directory with full info
        model_label = f'{brand_name}_{model_name}'.replace(' ', '_').replace('/', '_')
        model_output_dir = f'{output_dir}/{model_label}'
        os.makedirs(model_output_dir, exist_ok=True)
        
        # Copy images
        copied = 0
        for img_path in model_images[:500]:
            try:
                img = Image.open(img_path).convert('RGB')
                dst = f'{model_output_dir}/{copied:05d}.jpg'
                img.save(dst, 'JPEG', quality=95)
                copied += 1
            except:
                pass
        
        if copied >= 10:
            total_models += 1
            total_images += copied
            
            # Store metadata
            model_data.append({
                'label': model_label,
                'brand': brand_name,
                'model': model_name,
                'car_type': car_type_name,
                'max_speed': attrs.get('max_speed', 0),
                'displacement': attrs.get('displacement', 0),
                'doors': attrs.get('doors', 0),
                'seats': attrs.get('seats', 0),
                'images': copied
            })

# Save metadata
with open(f'{output_dir}/metadata.json', 'w') as f:
    json.dump(model_data, f, indent=2, ensure_ascii=False)

print(f'\nDataset prepared: {total_models} models, {total_images} images')
print(f'\nSample metadata:')
for m in model_data[:5]:
    print(f"  {m['brand']} {m['model']}: {m['car_type']}, {m['doors']}p, {m['seats']}s, {m['displacement']}L, {m['max_speed']}km/h")

In [ ]:
# Split dataset
random.seed(42)

for split in ['train', 'val', 'test']:
    os.makedirs(f'{output_dir}/{split}', exist_ok=True)

for model_name in os.listdir(output_dir):
    model_path = f'{output_dir}/{model_name}'
    if not os.path.isdir(model_path) or model_name in ['train', 'val', 'test']:
        continue
    
    images = [f for f in os.listdir(model_path) if f.endswith('.jpg')]
    random.shuffle(images)
    
    n = len(images)
    train_end = int(n * 0.7)
    val_end = int(n * 0.85)
    
    for i, img in enumerate(images):
        if i < train_end:
            split = 'train'
        elif i < val_end:
            split = 'val'
        else:
            split = 'test'
        
        src = f'{model_path}/{img}'
        dst = f'{output_dir}/{split}/{model_name}/{img}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
    
    shutil.rmtree(model_path)

# Count
for split in ['train', 'val', 'test']:
    split_path = f'{output_dir}/{split}'
    count = sum(len(os.listdir(f'{split_path}/{d}')) for d in os.listdir(split_path) if os.path.isdir(f'{split_path}/{d}'))
    print(f'{split}: {count} images')

In [ ]:
# Dataset class
class VehicleDataset(Dataset):
    def __init__(self, root, transform=None):
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        
        root = Path(root)
        for idx, class_dir in enumerate(sorted([d for d in root.iterdir() if d.is_dir()])):
            self.class_to_idx[class_dir.name] = idx
            for img in class_dir.glob('*.jpg'):
                self.images.append(img)
                self.labels.append(idx)
        
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.num_classes = len(self.class_to_idx)
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load
print('Loading datasets...')
train_ds = VehicleDataset(f'{output_dir}/train', train_transform)
val_ds = VehicleDataset(f'{output_dir}/val', val_transform)

print(f'Train: {len(train_ds)} images, {train_ds.num_classes} classes')
print(f'Val: {len(val_ds)} images')

In [ ]:
# Model
model = models.mobilenet_v3_large(pretrained=True)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, train_ds.num_classes)
model = model.to(device)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

print(f'Model: MobileNetV3-Large')
print(f'Classes: {train_ds.num_classes}')

In [ ]:
# Train
EPOCHS = 30
best_acc = 0.0
patience = 10
patience_counter = 0

print(f'Training for {EPOCHS} epochs...\n')
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.1f}%'})
    
    train_acc = 100. * correct / total
    scheduler.step()
    
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    
    print(f'Epoch {epoch+1}: Train={train_acc:.1f}% Val={val_acc:.1f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'num_classes': train_ds.num_classes,
            'class_to_idx': train_ds.class_to_idx,
            'idx_to_class': train_ds.idx_to_class,
        }, 'best_complete_vehicle.pth')
        print(f'  ✓ Saved (acc: {val_acc:.1f}%)')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

elapsed = time.time() - start_time
print(f'\n{"="*50}')
print(f'Training complete!')
print(f'Best accuracy: {best_acc:.1f}%')
print(f'Classes: {train_ds.num_classes}')
print(f'Time: {elapsed/60:.1f} minutes')
print(f'{"="*50}')

In [ ]:
# Export with metadata
!pip install -q onnx onnxscript

import torch.onnx

checkpoint = torch.load('best_complete_vehicle.pth', map_location='cpu')
model = models.mobilenet_v3_large(pretrained=False)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, checkpoint['num_classes'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Export ONNX
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'vehicle_complete_classifier.onnx',
    export_params=True, opset_version=11,
    input_names=['input'], output_names=['output'])
print('ONNX exported')

# Save complete metadata
with open(f'{output_dir}/metadata.json', 'r') as f:
    model_metadata = json.load(f)

# Create label mapping with metadata
labels = {}
for idx, class_name in checkpoint['idx_to_class'].items():
    # Find metadata for this class
    meta = next((m for m in model_metadata if m['label'] == class_name), {})
    labels[str(idx)] = {
        'label': class_name,
        'brand': meta.get('brand', class_name.split('_')[0]),
        'model': meta.get('model', class_name.split('_', 1)[1] if '_' in class_name else ''),
        'car_type': meta.get('car_type', 'Desconocido'),
        'doors': meta.get('doors', 0),
        'seats': meta.get('seats', 0),
        'displacement': meta.get('displacement', 0),
        'max_speed': meta.get('max_speed', 0)
    }

with open('vehicle_complete_labels.json', 'w') as f:
    json.dump(labels, f, indent=2, ensure_ascii=False)
print('Labels with metadata saved')

# Show sample
print('\nSample labels:')
for idx, data in list(labels.items())[:5]:
    print(f"  {data['brand']} {data['model']}: {data['car_type']}, {data['doors']}p, {data['seats']}s")

import os
for f in ['best_complete_vehicle.pth', 'vehicle_complete_classifier.onnx', 'vehicle_complete_labels.json']:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f'{f}: {size:.1f} MB')
print('\nDone! Download all 3 files.')